Шаг 0. Установка и импорты

In [2]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 80.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.51.1
    Uninstalling transformers-4.51.1:
      Successfully uninstalled transformers-4.51.1


In [3]:
!pip install transformers datasets


In [4]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 1.8 MB/s eta 0:00:00


In [5]:
import os
import random
import torch
import numpy as np

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForMaskedLM,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    DataCollatorForLanguageModeling,
    DataCollatorForTokenClassification,
)
import evaluate
from datasets import load_dataset, Dataset, DatasetDict

device = "cuda" if torch.cuda.is_available() else "cpu"


In [6]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

1. Обучите NER-модель

  * Загрузите набор данных Collection5 - 1 балл
  * Разбейте набор данных на train/test части
  * Дообучите модель rubert-tiny2 на train-части корпуса для решения NER-задачи, сделайте замеры качества NER-метрик до и после дообучения - 2 балла

In [7]:
!wget http://www.labinform.ru/pub/named_entities/collection5.zip -O collection5.zip
!unzip -o collection5.zip -d collection5/
!rm collection5.zip

--2025-04-17 08:40:23--  http://www.labinform.ru/pub/named_entities/collection5.zip
Resolving www.labinform.ru (www.labinform.ru)... 95.181.230.181
Connecting to www.labinform.ru (www.labinform.ru)|95.181.230.181|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1899530 (1.8M) [application/zip]
Saving to: ‘collection5.zip’

collection5.zip     100%[===================>]   1.81M  1.12MB/s    in 1.6s    

2025-04-17 08:40:26 (1.12 MB/s) - ‘collection5.zip’ saved [1899530/1899530]

Archive:  collection5.zip
   creating: collection5/Collection5/
  inflating: collection5/Collection5/001.ann  
  inflating: collection5/Collection5/001.txt  
  inflating: collection5/Collection5/002.ann  
  inflating: collection5/Collection5/002.txt  
  inflating: collection5/Collection5/003.ann  
  inflating: collection5/Collection5/003.txt  
  inflating: collection5/Collection5/004.ann  
  inflating: collection5/Collection5/004.txt  
  inflating: collection5/Collection5/005.ann  
  infl

In [8]:
!pip install corus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.7/83.7 kB 3.0 MB/s eta 0:00:00


In [9]:
import os
import re
from corus import load_ne5

dir_path = "collection5/Collection5"

In [10]:
def parse_ne5_record(record):
    """
    Извлекает токены и генерирует BIO-метки из объекта Ne5Markup.
    Использует record.text и record.spans.
    """
    text = record.text
    tokens = []
    offsets = []
    # Простой способ токенизации: разбиваем по непробельным последовательностям.
    for m in re.finditer(r'\S+', text):
        tokens.append(m.group())
        offsets.append((m.start(), m.end()))

    # Инициализируем все метки как "O"
    tags = ["O"] * len(tokens)

    # Обрабатываем каждую аннотацию (span)
    for span in record.spans:
        span_type = span.type  # например, "PER", "GEOPOLIT" и т.д.
        token_indices = []
        # Находим индексы токенов, оффсеты которых перекрываются со span
        for i, (start, end) in enumerate(offsets):
            if end > span.start and start < span.stop:
                token_indices.append(i)
        if token_indices:
            # Первый токен получает метку B-<span_type>, остальные I-<span_type>
            tags[token_indices[0]] = f"B-{span_type}"
            for i in token_indices[1:]:
                tags[i] = f"I-{span_type}"
    return tokens, tags




In [11]:
def read_collection5(dir_path):
    """
    Обходит все .txt файлы в папке dir_path (где лежат файлы Collection5),
    загружает объекты с помощью load_ne5 и для каждого объекта извлекает
    токены и метки через функцию parse_ne5_record.

    Возвращает:
      texts  - список списков токенов,
      labels - список списков BIO-меток.
    """
    records = load_ne5(dir_path)
    texts = []
    labels = []
    for record in records:
        tokens, tags = parse_ne5_record(record)
        if tokens:
            texts.append(tokens)
            labels.append(tags)
    return texts, labels


In [12]:
# Шаг 1.1. Загрузка через corus.load_ne5

dir_path = "collection5/Collection5"  # убедитесь, что этот путь корректен
collection_texts, collection_labels = read_collection5(dir_path)
print("Считано предложений:", len(collection_texts))
if collection_texts:
    print("Пример токенов:", collection_texts[0])
    print("Пример меток: ", collection_labels[0])

Считано предложений: 1000
Пример токенов: ['Распределены', 'кураторы', 'госпрограмм', 'в', 'правительстве', 'РФ', 'Больше', 'других', '"работы"', 'досталось', 'Игорю', 'Шувалову,', 'Ольге', 'Голодец,', 'Аркадию', 'Дворковичу', 'и', 'Дмитрию', 'Рогозину.', 'Премьер', 'Дмитрий', 'Медведев', 'распределил', 'между', 'своими', 'заместителями', 'функции', 'кураторов', 'реализации', 'госпрограмм,', 'сообщается', 'на', 'сайте', 'правительства', 'России.', 'Больше', 'других', '"работы"', 'досталось', 'Игорю', 'Шувалову,', 'Ольге', 'Голодец,', 'Аркадию', 'Дворковичу', 'и', 'Дмитрию', 'Рогозину.', 'Шувалову', 'и', 'Голодец', 'назначено', 'по', 'восемь', 'программ,', 'а', 'Дворкович', 'и', 'Рогозин', 'будут', 'курировать', 'по', '11', 'проектов.', 'Как', 'уточняется', 'на', 'сайте', 'кабмина,', 'в', 'частности', 'вице-премьер', 'по', 'оборонной', 'промышленности', 'Рогозин', 'будет', 'отвечать', 'за', 'противодействие', 'незаконному', 'обороту', 'наркотиков,', 'развитие', 'судостроения', 'и', 'кос

In [14]:
# Преобразуем данные в Hugging Face Dataset
dataset_dict = {
    "tokens": collection_texts,
    "ner_tags": collection_labels
}
dataset = Dataset.from_dict(dataset_dict)
print(dataset)

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 1000
})


In [15]:
# 1.2. Разбиваем на train/test
ds_split = dataset.train_test_split(test_size=0.2, seed=SEED)
train_dataset = ds_split["train"]
test_dataset  = ds_split["test"]

print("Train:", len(train_dataset), "Test:", len(test_dataset))


Train: 800 Test: 200


In [16]:
# Шаг 1.3.
# Подготовка к обучению rubert-tiny2 на NER

from transformers import AutoTokenizer, AutoModelForTokenClassification
import evaluate

# Токенизатор
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")

# Список уникальных тегов
unique_tags = set(tag for tags in train_dataset["ner_tags"] for tag in tags)
tag2id = {tag: i for i, tag in enumerate(sorted(unique_tags))}
id2tag = {i: tag for tag, i in tag2id.items()}
num_labels = len(tag2id)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True
    )
    new_labels = []
    for i, tags in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned_tags = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                aligned_tags.append(-100)
            elif word_id != prev_word_id:
                # первый сабтокен
                aligned_tags.append(tag2id[ tags[word_id] ])
            else:
                # остальные сабтокены того же слова
                aligned_tags.append(-100)
            prev_word_id = word_id
        new_labels.append(aligned_tags)
    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

train_dataset = train_dataset.map(tokenize_and_align_labels, batched=True)
test_dataset  = test_dataset.map(tokenize_and_align_labels,  batched=True)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [17]:
# Модель
model = AutoModelForTokenClassification.from_pretrained(
    "cointegrated/rubert-tiny2",
    num_labels=num_labels,
    id2label=id2tag,
    label2id=tag2id
).to(device)

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=7fea72330796b691db6cf5ea8ffdd3b439121352125cc8a17e2a5056bd0a29b6
  Stored in directory: /root/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
Successfully built seqeval


In [19]:
# seqeval
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    import numpy as np
    predictions, references = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_preds = []
    true_labels = []
    for pred, ref in zip(predictions, references):
        temp_preds, temp_labels = [], []
        for p, r in zip(pred, ref):
            if r != -100:
                temp_preds.append(id2tag[p])
                temp_labels.append(id2tag[r])
        true_preds.append(temp_preds)
        true_labels.append(temp_labels)
    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }

In [20]:
test_dataset = test_dataset.filter(lambda x: len(x["input_ids"]) > 0)

Filter:   0%|          | 0/200 [00:00<?, ? examples/s]

In [21]:
from transformers import DataCollatorForTokenClassification, TrainingArguments, Trainer
import os
os.environ["WANDB_DISABLED"] = "true"

data_collator = DataCollatorForTokenClassification(tokenizer)

# Проверим метрики "до" (без обучения)
args_eval = dict(metric_key_prefix="test")
trainer_eval_before = Trainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
res_before = trainer_eval_before.evaluate(test_dataset)
print("METRICS BEFORE FINE-TUNING:", res_before)

<ipython-input-21-68d87ff0c02c>:9: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_eval_before = Trainer(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


METRICS BEFORE FINE-TUNING: {'eval_loss': 2.690098762512207, 'eval_model_preparation_time': 0.0008, 'eval_precision': 0.014074982036434338, 'eval_recall': 0.06473561430793157, 'eval_f1': 0.023122591396729508, 'eval_accuracy': 0.0165508109897385, 'eval_runtime': 6.6098, 'eval_samples_per_second': 30.258, 'eval_steps_per_second': 3.782}


In [22]:
# Шаг 1.3. fine-tuning

training_args = TrainingArguments(
    output_dir="rubert_tiny2_ner",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    seed=SEED,
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-22-5f8e231a729b>:13: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [23]:
trainer.train()

Step,Training Loss
50,1.565400
100,0.747300
150,0.594200


TrainOutput(global_step=150, training_loss=0.96894957224528, metrics={'train_runtime': 798.4939, 'train_samples_per_second': 3.006, 'train_steps_per_second': 0.188, 'total_flos': 27421962578784.0, 'train_loss': 0.96894957224528, 'epoch': 3.0})

In [24]:
# Оценка после обучения
res_after = trainer.evaluate(test_dataset)
print("METRICS AFTER FINE-TUNING:", res_after)

METRICS AFTER FINE-TUNING: {'eval_loss': 0.553072452545166, 'eval_precision': 0.2349743279975838, 'eval_recall': 0.15124416796267495, 'eval_f1': 0.18403311649911294, 'eval_accuracy': 0.8511845651865513, 'eval_runtime': 7.8884, 'eval_samples_per_second': 25.354, 'eval_steps_per_second': 1.648, 'epoch': 3.0}


/usr/local/lib/python3.11/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


До дообучения модель выдавала eval_loss ~ 2.69 с низкими метриками (precision ~1.4%, recall ~6.5%, f1 ~2.3% и accuracy ~1.7%), что походит на случайное поведение.

После Файтюна eval_loss снизилась до 0.55 (accuracy 85.1%, precision≈23.5%, recall ≈15.1%, f1≈18.4%), что говорит нам о повышение способности модели различать классы.

# 2. Попробуйте улучшить качество модели следующими способами:
  * Предварительно дообучите на train-части в MLM режиме, а потом дообучите на NER-задачу - 2 балла
  * Сгенерируйте синтетическую разметку* подходящего**, на ваш взгляд, новостного корпуса большой и умной моделью для русскоязычного NER***, а затем использовав ее для дообучения rubert-tiny2 вместе с основным набором данных - 2 балла

In [25]:
from transformers import AutoModelForMaskedLM, DataCollatorForLanguageModeling, TrainingArguments, Trainer
from datasets import Dataset

In [26]:
# Создадим MLM-датасет: для каждого примера объединим токены в строку.
def join_tokens(example):
    example["text"] = " ".join(example["tokens"])
    return example

mlm_dataset = train_dataset.map(join_tokens)

mlm_dataset = mlm_dataset.remove_columns(["ner_tags", "tokens", "labels"])

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [27]:
# Задаём block_size для обрезки длинных примеров (опционально)
block_size = 128
def tokenize_mlm(examples):
    return tokenizer(examples["text"], truncation=True, max_length=block_size, padding="max_length")

mlm_dataset = mlm_dataset.map(tokenize_mlm, batched=True)

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [28]:
# Data collator для MLM
mlm_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

In [29]:
# Загружаем модель для MLM
mlm_model = AutoModelForMaskedLM.from_pretrained("cointegrated/rubert-tiny2").to(device)

In [30]:
mlm_training_args = TrainingArguments(
    output_dir="rubert_tiny2_mlm",
    learning_rate=2e-5,
    num_train_epochs=1,
    per_device_train_batch_size=16,
    logging_steps=50,
    seed=SEED,
)


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [31]:
mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_training_args,
    train_dataset=mlm_dataset,
    data_collator=mlm_collator,
    tokenizer=tokenizer,
)

<ipython-input-31-bedaea9934fe>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  mlm_trainer = Trainer(


In [32]:
print("Ключи в датасете:", mlm_dataset.column_names)

Ключи в датасете: ['input_ids', 'token_type_ids', 'attention_mask', 'text']


In [33]:
mlm_trainer.train()

Step,Training Loss
50,3.315400


TrainOutput(global_step=50, training_loss=3.3154217529296877, metrics={'train_runtime': 114.4648, 'train_samples_per_second': 6.989, 'train_steps_per_second': 0.437, 'total_flos': 1526344089600.0, 'train_loss': 3.3154217529296877, 'epoch': 1.0})

In [34]:
# Сохраняем дообученные веса и загружаем их в NER модель:
mlm_model.save_pretrained("rubert_tiny2_mlm_finetuned")


In [35]:
# Загружаем rubert-tiny2 в режиме NER с дообученными весами
from transformers import AutoModelForTokenClassification
ner_model_mlm = AutoModelForTokenClassification.from_pretrained(
    "rubert_tiny2_mlm_finetuned",
    num_labels=num_labels,
    id2label=id2tag,
    label2id=tag2id,
).to(device)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at rubert_tiny2_mlm_finetuned and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [36]:
# Затем выполняем finetuning на NER-данных
ner_trainer_mlm = Trainer(
    model=ner_model_mlm,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)



<ipython-input-36-6b43a35e5dc4>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  ner_trainer_mlm = Trainer(


In [37]:
ner_trainer_mlm.train()

Step,Training Loss
50,1.475600
100,0.731700
150,0.591200


TrainOutput(global_step=150, training_loss=0.9328338877360026, metrics={'train_runtime': 797.3655, 'train_samples_per_second': 3.01, 'train_steps_per_second': 0.188, 'total_flos': 27421962578784.0, 'train_loss': 0.9328338877360026, 'epoch': 3.0})

In [38]:
res_after_mlm = ner_trainer_mlm.evaluate(test_dataset)
print("METRICS AFTER MLM-PRETRAINING + NER FINETUNING:", res_after_mlm)

METRICS AFTER MLM-PRETRAINING + NER FINETUNING: {'eval_loss': 0.5475249290466309, 'eval_precision': 0.33891702376314764, 'eval_recall': 0.16912908242612754, 'eval_f1': 0.22565166645052523, 'eval_accuracy': 0.8522958339244338, 'eval_runtime': 7.9591, 'eval_samples_per_second': 25.128, 'eval_steps_per_second': 1.633, 'epoch': 3.0}


/usr/local/lib/python3.11/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


* Сгенерируйте синтетическую разметку* подходящего**, на ваш взгляд, новостного корпуса большой и умной моделью для русскоязычного NER***, а затем использовав ее для дообучения rubert-tiny2 вместе с основным набором данных - 2 балла

In [ ]:
!pip install navec slovnet

In [ ]:
!pip install navec slovnet razdel datasets transformers evaluate

In [ ]:
!wget https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar

In [ ]:
!wget https://storage.yandexcloud.net/natasha-slovnet/packs/slovnet_ner_news_v1.tar

In [48]:
from navec import Navec
from slovnet import NER
from razdel import tokenize
from datasets import load_dataset, Dataset
import torch, random, numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments, Trainer,
    DataCollatorForTokenClassification
)
import evaluate


SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:

navec = Navec.load("navec_news_v1_1B_250K_300d_100q.tar")
ner_sl = NER.load("slovnet_ner_news_v1.tar")

ner_sl.navec(navec)

In [ ]:
!wget https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz

In [52]:
from corus import load_lenta
path = "lenta-ru-news.csv.gz"
records = load_lenta(path)

texts_large = [next(records).text for _ in range(10_000)]


In [53]:
# Функция преобразования markup.spans → токены + BIO‑теги
import re
def markup_to_bio(markup):
    text = markup.text
    # 1) Разбиваем на токены «в лоб»
    toks, offs = [], []
    for m in re.finditer(r'\S+', text):
        toks.append(m.group()); offs.append((m.start(), m.end()))
    tags = ["O"] * len(toks)
    # 2) Для каждого span ставим B‑/I‑метки
    for sp in markup.spans:
        idxs = [i for i,(s,e) in enumerate(offs) if e>sp.start and s<sp.stop]
        if not idxs: continue
        tags[idxs[0]] = f"B-{sp.type}"
        for i in idxs[1:]:
            tags[i] = f"I-{sp.type}"
    return toks, tags

In [54]:
# Генерируем синтетические примеры
synt_toks, synt_tags = [], []
for txt in texts_large:
    m = ner_sl(txt)
    toks, tags = markup_to_bio(m)
    if toks:
        synt_toks.append(toks)
        synt_tags.append(tags)

In [56]:
# Объединяем с оригинальным train_dataset
orig_toks = train_dataset["tokens"]
orig_tags = train_dataset["ner_tags"]
all_toks  = orig_toks + synt_toks
all_tags  = orig_tags + synt_tags

In [57]:
from datasets import Dataset
combined = Dataset.from_dict({"tokens": all_toks, "ner_tags": all_tags})
ds = combined.train_test_split(test_size=0.2, seed=SEED)
train_comb, test_comb = ds["train"], ds["test"]

In [58]:
# Выравниваем токены и метки под rubert-tiny2
train_comb = train_comb.map(tokenize_and_align_labels, batched=True)
test_comb  = test_comb.map(tokenize_and_align_labels,  batched=True)

Map:   0%|          | 0/8640 [00:00<?, ? examples/s]

Map:   0%|          | 0/2160 [00:00<?, ? examples/s]

In [59]:
# Загружаем rubert-tiny2 для NER
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification

tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")
model_syn = AutoModelForTokenClassification.from_pretrained(
    "cointegrated/rubert-tiny2",
    num_labels=num_labels,
    id2label=id2tag,
    label2id=tag2id
).to(device)


Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [62]:
data_collator = DataCollatorForTokenClassification(tokenizer)
seqeval       = evaluate.load("seqeval")

def compute_metrics(p):
    import numpy as np
    preds, refs = p
    preds = np.argmax(preds, axis=2)
    true_p, true_r = [], []
    all_preds, all_labels = [], []
    for pred, lab in zip(preds, refs):
        pp, ll = [], []
        for p_i, l_i in zip(pred, lab):
            if l_i != -100:
                pp.append(id2tag[p_i]); ll.append(id2tag[l_i])
        all_preds.append(pp); all_labels.append(ll)
    res = seqeval.compute(predictions=all_preds, references=all_labels)
    return {
        "precision": res["overall_precision"],
        "recall":    res["overall_recall"],
        "f1":        res["overall_f1"],
        "accuracy":  res["overall_accuracy"],
    }

In [70]:
# Fine‑tuning на комбинированном датасете
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="rubert_tiny2_ner_synth",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    seed=SEED,
    logging_steps=100,
    report_to=[]
)
trainer_syn = Trainer(
    model=model_syn,
    args=training_args,
    train_dataset=train_comb,
    eval_dataset=test_comb,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


trainer_syn.train()

res_syn = trainer_syn.evaluate(test_comb)
print("METRICS AFTER SYNTHETIC LABELING + FINETUNING:", res_syn)

<ipython-input-70-a3915e10865b>:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_syn = Trainer(


Step,Training Loss
100,0.934900
200,0.278300
300,0.200900
400,0.158100
500,0.133200
600,0.114500
700,0.100600
800,0.094300
900,0.087400
1000,0.082400


/usr/local/lib/python3.11/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


METRICS AFTER SYNTHETIC LABELING + FINETUNING: {'eval_loss': 0.06469561159610748, 'eval_precision': 0.8241717929905751, 'eval_recall': 0.8878047297708067, 'eval_f1': 0.8548056661203466, 'eval_accuracy': 0.9813552363477855, 'eval_runtime': 58.3699, 'eval_samples_per_second': 37.005, 'eval_steps_per_second': 2.313, 'epoch': 3.0}


In [74]:
# ШАГ 3
import pandas as pd

def extract(res):
    return {
        "Loss":      res["eval_loss"],
        "Precision": res["eval_precision"],
        "Recall":    res["eval_recall"],
        "F1":        res["eval_f1"],
        "Accuracy":  res["eval_accuracy"],
    }

# Собираем результаты
results = [
    {"Approach": "Baseline (до обучения)", **extract(res_before)},
    {"Approach": "Fine‑tuning (NER)", **extract(res_after)},
    {"Approach": "MLM‑предобучение + NER", **extract(res_after_mlm)},
    {"Approach": "Синтетическая разметка + NER", **extract(res_syn)},
]

# Делаем DataFrame и выводим
df = pd.DataFrame(results)[["Approach", "Loss", "Precision", "Recall", "F1", "Accuracy"]]
print(df.to_string(index=False, float_format="{:0.4f}".format))


                    Approach   Loss  Precision  Recall     F1  Accuracy
      Baseline (до обучения) 2.6901     0.0141  0.0647 0.0231    0.0166
           Fine‑tuning (NER) 0.5531     0.2350  0.1512 0.1840    0.8512
      MLM‑предобучение + NER 0.5475     0.3389  0.1691 0.2257    0.8523
Синтетическая разметка + NER 0.0647     0.8242  0.8878 0.8548    0.9814


Без дообучения модель почти «угадывала» – F1 ≈ 2%, точность около 1,7%.

Обычный fine‑tuning на Collection5 повысил F1 до ≈ 18% и accuracy до ≈ 85%.

MLM-предобучение + NER дало ещё небольшой прирост: F1 ≈ 22%, accuracy ≈ 85%.

Синтетическая разметка + NER на большом корпусе резко улучшила результаты – F1 ≈ 85% и accuracy ≈ 98%.